# Part 1：收益建模（H / B / BQ，拟合与预测）

函数定义在 `src/return_model.py`。调参（`tune_alpha` 按验证期 MSE）在 `05_pipeline`。ver3：训练目标为横截面相对收益，回退股票预测 0。

In [ ]:
import sys
sys.path.insert(0, 'e:/4020prj/code/ver3')
from src.return_model import *
from src.config import CONFIG
import numpy as np, pandas as pd


**函数与计算方式**

- `historical_mean_prediction(R_window)`：训练窗口内每股票月收益均值（绝对收益，作 H 基准）。
- `fit_ridge(X, y, w, alpha)`：合并岭回归 `min ||y-Xβ||² + α||β||²`。
- `fit_predict_return(train_panel, X_score, alpha, fields, fallback_mu=0.0)`：训练并预测目标截面相对收益，回退股票填 `fallback_mu`（默认 0 = 零相对收益）。
- `cross_sectional_spearman(mu, R)`：截面 Spearman IC。
- `relative_mse(mu, R)`：预测相对收益 vs 真实相对收益（R − 池等权均值）的 MSE。

合成数据演示：

In [ ]:
rng = np.random.default_rng(0)
R_win = pd.DataFrame(rng.normal(0.005, 0.06, size=(120, 50)))
print('H 预测 std:', round(float(historical_mean_prediction(R_win).std()), 4))
n = len(CONFIG['BASE_FIELDS'])
X = pd.DataFrame(rng.normal(size=(500, n)), columns=CONFIG['BASE_FIELDS'])
m = fit_ridge(X, pd.Series(rng.normal(size=500)), pd.Series(np.full(500, 1/500)), 10.0)
print('岭系数前3:', m.coef_[:3].round(3))
print('示例 IC:', round(cross_sectional_spearman(pd.Series(rng.normal(size=50)), pd.Series(rng.normal(size=50))), 3))
